# NOT training with 2D Synthetic Data

In [ ]:
import os, sys
sys.path.append("..")

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from IPython.display import clear_output

device = "cuda" if torch.cuda.is_available() else "cpu"

from src.models import Transport, Critic
from src.utils import cost, show_mapping , grad_norm, cosine_lr, plot_3d_function
from src.cost import compute_transport_cost, sinkhorn_w2, compute_w2_sinkhorn
from src.train import train_extragradient


## Distributions

In [ ]:
# source distribution μ
def sample_mu(batch_size, device="cpu"):
    x = 2 * torch.rand(batch_size, 1, device=device) - 1  # [-1,1]
    y = torch.zeros(batch_size, 1, device=device)         # y = 0
    return torch.cat([x, y], dim=1)


# target distribution ν
def sample_nu(batch_size, device="cpu"):
    x = torch.zeros(batch_size, 1, device=device)         # x = 0
    y = 2 * torch.rand(batch_size, 1, device=device) - 1  # [-1,1]
    return torch.cat([x, y], dim=1)

In [ ]:
def sample_nu(batch_size, device="cpu"):
    x = 2 * torch.rand(batch_size, 1, device=device) - 1  # [-1,1]
    y = torch.ones(batch_size, 1, device=device)         # y = 1
    return torch.cat([x, y], dim=1)

In [ ]:
def sample_nu(batch_size, device="cpu"):
    x = 2 * torch.rand(batch_size, 1, device=device) - 1  # [-1,1]

    sign = torch.randint(0, 2, (batch_size, 1), device=device) * 2 - 1
    y = sign.float()  # y = +1 or -1

    return torch.cat([x, y], dim=1)

In [ ]:
def sample_nu(batch_size, device="cpu", n_grid=5):
    grid = torch.linspace(-1, 1, 2*n_grid-1, device=device)  # grid points
    idx = 2*torch.randint(1, n_grid, (batch_size,), device=device)-1

    x = grid[idx].unsqueeze(1)
    y = 2 * torch.rand(batch_size, 1, device=device) - 1

    return torch.cat([x, y], dim=1)

In [ ]:
def sample_mu(batch_size, device="cpu", n_grid=5):
    grid = torch.linspace(-1, 1, 2*n_grid-1, device=device)  # grid points
    idx = 2*torch.randint(1, n_grid, (batch_size,), device=device)-1

    y = grid[idx].unsqueeze(1)
    x = 2 * torch.rand(batch_size, 1, device=device) - 1

    return torch.cat([x, y], dim=1)

In [ ]:
T = Transport().to(device)
f = Critic().to(device)

## Extragradient Training

In [ ]:
history = train_extragradient(
    T, f, sample_mu, sample_nu, cost,
    n_steps=200000, lr_T=1e-3 / 2, lr_f=1e-3 / 2,
    batch_size_x=1024, batch_size_y=1024,
    callback=lambda step, T, f: show_mapping(T, sample_mu, sample_nu, f=f, contour=False),
)

## Plot Results

In [ ]:
show_mapping(T,sample_mu,sample_nu, option = True)

In [ ]:
plot_3d_function(f)

In [ ]:
# cost
print("cost:")
print("\t".join(str(compute_transport_cost(T, sample_mu, n_samples=2048))
                for _ in range(10)))

# w2
print("D_map:")
print("\t".join(str(compute_w2_sinkhorn(T, sample_mu, sample_nu,
                                        n_samples=2048))
                for _ in range(10)))
